In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load a pre-trained language model (GPT-2) for perplexity calculations
tokenizer = AutoTokenizer.from_pretrained("gpt2")  # GPT-2 tokenizer (English)
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()
if torch.cuda.is_available():
    model.cuda()

def calculate_perplexity(text: str) -> float:
    """
    Calculate the perplexity of the given text using GPT-2.
    Perplexity = exp(cross-entropy loss of the language model on the text).
    """
    # Tokenize input with GPT-2 tokenizer. Truncate long text to model's max length for simplicity.
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=model.config.n_positions)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k,v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs['input_ids'])
        # outputs.loss is the mean cross-entropy loss over input tokens
        loss = outputs.loss.item()
        perplexity = np.exp(loss)
        return perplexity

# Example usage:
ppl = calculate_perplexity("This is a sample sentence for perplexity test.")
print("Perplexity:", ppl)
